In [2]:
!nvidia-smi
!find /kaggle/input -maxdepth 4

Wed Oct 21 08:10:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%%writefile sgn.py
#!/usr/bin/env python
"""SpectralGate-Net: minimal study on simulated thermal crossover built from COD10K.

  python sgn.py train --mode full   --root COD10K-v3
  python sgn.py train --mode fixed  --root COD10K-v3
  python sgn.py train --mode concat --root COD10K-v3
  python sgn.py train --mode rgb    --root COD10K-v3
  python sgn.py report                      # writes LaTeX tables + PDF plots

  python sgn.py train --mode full --smoke   # 1-minute CPU/GPU sanity check, synthetic data
"""
import argparse, glob, json, math, os, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as Fn
import torchvision
from torchvision.ops import deform_conv2d

DELTAS = [8, 4, 2, 1, 0]            # test-time target-minus-background temperature (K)
TRAIN_D = [0, 0.5, 1, 2, 4, 8]      # training-time contrast values (sampled per image)

# ------------------------------------------------------------------ thermal simulation
H_PL, C_PL, K_PL = 6.626e-34, 2.998e8, 1.381e-23


def planck_band(T):
    """Band-integrated (8-14 um) blackbody radiance, arbitrary units. T in kelvin."""
    lam = torch.linspace(8e-6, 14e-6, 13, dtype=torch.float64, device=T.device).view(1, -1, 1, 1)
    T = T.double()
    w = 2 * H_PL * C_PL ** 2 / lam ** 5 / torch.expm1(H_PL * C_PL / (lam * K_PL * T))
    return w.sum(1, keepdim=True)


def smooth_noise(B, H, W, res, dev, gen=None):
    n = torch.randn(B, 1, res, res, device=dev, generator=gen)
    return Fn.interpolate(n, size=(H, W), mode="bicubic", align_corners=False)


def gauss_blur(x, sigma=1.0, k=5):
    ax = torch.arange(k, device=x.device, dtype=x.dtype) - k // 2
    g = torch.exp(-ax ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    x = Fn.conv2d(Fn.pad(x, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k))
    return Fn.conv2d(Fn.pad(x, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1))


@torch.no_grad()
def simulate_lwir(rgb, mask, delta, gen=None, T0=295.0, netd=0.05):
    """rgb (B,3,H,W) in [0,1]; mask (B,1,H,W) in {0,1}; delta (B,) kelvin -> LWIR (B,1,H,W) in [0,1].
    Tbg = T0 + 2K * spatially-correlated field + 2K * (luminance - 0.5)   [scene-dependent term]
    Ttarget = Tbg + delta inside mask; Gaussian PSF (sigma 1 px); NETD 50 mK; fixed 280-320 K -> 8-bit range."""
    B, _, H, W = rgb.shape
    lum = rgb.mean(1, keepdim=True)
    T = T0 + 2.0 * smooth_noise(B, H, W, 12, rgb.device, gen) + 2.0 * (lum - 0.5)
    T = T + delta.view(B, 1, 1, 1).to(rgb.device) * mask
    T = gauss_blur(T)
    T = T + netd * torch.randn(T.shape, device=T.device, generator=gen)
    lo = planck_band(torch.full((1, 1, 1, 1), 280.0, device=T.device))
    hi = planck_band(torch.full((1, 1, 1, 1), 320.0, device=T.device))
    g = ((planck_band(T) - lo) / (hi - lo)).clamp(0, 1).float()
    return torch.round(g * 255) / 255


def shadow_rgb(rgb, gen=None):
    """Dappled-shadow darkening of the visible image."""
    B, _, H, W = rgb.shape
    s = torch.sigmoid(2.0 * smooth_noise(B, H, W, 8, rgb.device, gen))
    return rgb * (0.25 + 0.75 * s)


# ------------------------------------------------------------------ Haar DWT / IDWT
def dwt(x):
    a, b, c, d = x[..., 0::2, 0::2], x[..., 0::2, 1::2], x[..., 1::2, 0::2], x[..., 1::2, 1::2]
    return (a + b + c + d) / 2, (a - b + c - d) / 2, (a + b - c - d) / 2, (a - b - c + d) / 2


def idwt(ll, lh, hl, hh):
    a = (ll + lh + hl + hh) / 2; b = (ll - lh + hl - hh) / 2
    c = (ll + lh - hl - hh) / 2; d = (ll - lh - hl + hh) / 2
    B, C, H, W = ll.shape
    return Fn.pixel_shuffle(torch.stack([a, b, c, d], 2).reshape(B, C * 4, H, W), 2)


# ------------------------------------------------------------------ gate
class SoftEntropy(nn.Module):
    def __init__(self, M=16, tau=0.2):
        super().__init__()
        self.e = nn.Parameter(torch.linspace(-1, 1, M)); self.tau = tau; self.M = M

    def forward(self, t):                      # t in [-1,1], (B,C,H,W)
        v = t.float().reshape(t.shape[0], -1, 1)
        p = torch.softmax(-(v - self.e.view(1, 1, -1)) ** 2 / (2 * self.tau ** 2), dim=-1)
        ph = p.mean(1)
        return -(ph * torch.log(ph + 1e-7)).sum(-1) / math.log(self.M)   # normalised to [0,1]


class Gate(nn.Module):
    def __init__(self, extra=False):
        super().__init__()
        self.ent = SoftEntropy(); self.extra = extra
        self.mlp = nn.Sequential(nn.Linear(3 if extra else 2, 16), nn.GELU(), nn.Linear(16, 1))
        nn.init.zeros_(self.mlp[2].weight); nn.init.zeros_(self.mlp[2].bias)   # start at alpha = 0.5

    def forward(self, f, raw_std=None):
        t = torch.tanh(f.float())
        var = t.var(dim=(2, 3)).mean(1) * 10.0
        z = [var, self.ent(t)]
        if self.extra:
            z.append(raw_std.float())
        return torch.sigmoid(self.mlp(torch.stack(z, 1))).squeeze(1)      # (B,)


# ------------------------------------------------------------------ fusion blocks
class WaveFuse(nn.Module):
    def __init__(self, C, mode, heads=4, extra=False):
        super().__init__()
        self.mode = mode
        self.off = nn.Conv2d(2 * C, 27, 3, padding=1)
        nn.init.zeros_(self.off.weight); nn.init.zeros_(self.off.bias)
        self.w_ll = nn.Parameter(torch.empty(C, C, 3, 3)); nn.init.kaiming_uniform_(self.w_ll, a=math.sqrt(5))
        self.w_hf = nn.Parameter(torch.empty(C, C, 3, 3)); nn.init.kaiming_uniform_(self.w_hf, a=math.sqrt(5))
        self.g_rgb, self.g_ir = Gate(extra), Gate(extra)
        mk = lambda: nn.ModuleList([nn.MultiheadAttention(C, heads, batch_first=True) for _ in range(3)])
        self.r2i, self.i2r = mk(), mk()                       # separate projections per direction
        self.ln = nn.ModuleList([nn.LayerNorm(C) for _ in range(3)])

    def forward(self, fr, fi, raw_r=None, raw_i=None):
        llr, lhr, hlr, hhr = dwt(fr)
        lli, lhi, hli, hhi = dwt(fi)
        with torch.autocast(device_type=fr.device.type, enabled=False):      # deform conv in fp32
            o = self.off(torch.cat([llr, lli], 1).float())
            off, m = o[:, :18], 2 * torch.sigmoid(o[:, 18:])
            lli = deform_conv2d(lli.float(), off, self.w_ll, padding=1, mask=m)
            lhi, hli, hhi = [deform_conv2d(x.float(), off, self.w_hf, padding=1, mask=m) for x in (lhi, hli, hhi)]
        B = fr.shape[0]
        if self.mode == "fixed":
            a_r = a_i = torch.full((B,), 0.5, device=fr.device)
        else:
            a_r, a_i = self.g_rgb(llr, raw_r), self.g_ir(lli, raw_i)
        self.gates = (a_r, a_i)
        ar4, ai4 = a_r.view(-1, 1, 1, 1), a_i.view(-1, 1, 1, 1)
        ll = ar4 * llr + ai4 * lli
        outs = []
        for k, (X, Y) in enumerate([(lhr, lhi), (hlr, hli), (hhr, hhi)]):
            Bc, C, h, w = X.shape
            xt, yt = X.flatten(2).transpose(1, 2), Y.to(X.dtype).flatten(2).transpose(1, 2)
            A1 = self.r2i[k](xt, yt, yt, need_weights=False)[0]                # Q from RGB, K/V from LWIR
            A2 = self.i2r[k](yt, xt, xt, need_weights=False)[0]
            o_ = a_i.view(-1, 1, 1) * A1 + a_r.view(-1, 1, 1) * A2 + self.ln[k](xt * yt)
            outs.append(o_.transpose(1, 2).reshape(Bc, C, h, w))
        return idwt(ll, *outs)


class GatedSum(nn.Module):
    def __init__(self, C, mode, extra=False):
        super().__init__()
        self.mode = mode; self.g_rgb, self.g_ir = Gate(extra), Gate(extra)

    def forward(self, fr, fi, raw_r=None, raw_i=None):
        B = fr.shape[0]
        if self.mode == "fixed":
            a_r = a_i = torch.full((B,), 0.5, device=fr.device)
        else:
            a_r, a_i = self.g_rgb(fr, raw_r), self.g_ir(fi, raw_i)
        self.gates = (a_r, a_i)
        return a_r.view(-1, 1, 1, 1) * fr + a_i.view(-1, 1, 1, 1) * fi


class ConcatFuse(nn.Module):
    def __init__(self, C):
        super().__init__(); self.c = nn.Sequential(nn.Conv2d(2 * C, C, 1), nn.BatchNorm2d(C), nn.ReLU())

    def forward(self, fr, fi, *_):
        return self.c(torch.cat([fr, fi], 1))


# ------------------------------------------------------------------ network
def run_backbone(m, x):
    x = m.maxpool(m.relu(m.bn1(m.conv1(x)))); x = m.layer1(x)
    f3 = m.layer2(x); f4 = m.layer3(f3); f5 = m.layer4(f4)
    return f3, f4, f5


class Net(nn.Module):
    def __init__(self, mode, pretrained=True, extra=False):
        super().__init__()
        self.mode, self.extra = mode, extra
        w = "IMAGENET1K_V1" if pretrained else None
        self.b_rgb = torchvision.models.resnet18(weights=w)
        if mode != "rgb":
            self.b_ir = torchvision.models.resnet18(weights=w)
        ch = (128, 256, 512)
        if mode in ("full", "fixed"):
            self.fuse = nn.ModuleList([WaveFuse(128, mode, extra=extra), WaveFuse(256, mode, extra=extra),
                                       GatedSum(512, mode, extra=extra)])
        elif mode == "concat":
            self.fuse = nn.ModuleList([ConcatFuse(c) for c in ch])
        self.lat = nn.ModuleList([nn.Conv2d(c, 64, 1) for c in ch])
        self.out = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.Conv2d(64, 1, 1))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, rgb, ir):
        size = rgb.shape[-2:]
        fr = run_backbone(self.b_rgb, (rgb - self.mean) / self.std)
        if self.mode == "rgb":
            fused, gates = fr, []
        else:
            fi = run_backbone(self.b_ir, ((ir - 0.45) / 0.225).expand(-1, 3, -1, -1))
            raw_r = rgb.flatten(1).std(1) * 10 if self.extra else None
            raw_i = ir.flatten(1).std(1) * 10 if self.extra else None
            fused = [f(a, b, raw_r, raw_i) for f, a, b in zip(self.fuse, fr, fi)]
            gates = [f.gates for f in self.fuse] if self.mode in ("full", "fixed") else []
        p = [l(f) for l, f in zip(self.lat, fused)]
        up = lambda t: Fn.interpolate(t, size=p[0].shape[-2:], mode="bilinear", align_corners=False)
        x = p[0] + up(p[1]) + up(p[2])
        return Fn.interpolate(self.out(x), size=size, mode="bilinear", align_corners=False), gates


# ------------------------------------------------------------------ data
def load_split(root, split, size):
    """Finds COD10K camouflaged (COD10K-CAM-*) images + GT_Object masks anywhere under root.
    Non-camouflaged / background images are skipped (they have no camouflage target)."""
    cache = f"cache_{split}_{size}.npz"
    if os.path.exists(cache):
        d = np.load(cache); return d["x"], d["m"]
    from PIL import Image
    from concurrent.futures import ThreadPoolExecutor
    in_split = lambda p: split.lower() in [c.lower() for c in p.split(os.sep)]
    ims = sorted(p for p in glob.glob(os.path.join(root, "**", "*.jpg"), recursive=True)
                 if in_split(p) and os.path.basename(p).startswith("COD10K-CAM-"))
    gts = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(root, "**", "*.png"), recursive=True)
           if in_split(p) and "object" in p.lower() and "instance" not in p.lower()}
    ims = [p for p in ims if os.path.splitext(os.path.basename(p))[0] in gts]
    assert ims, f"no COD10K-CAM images+GT_Object masks found for split {split} under {root}"
    print(f"{split}: {len(ims)} camouflaged images", flush=True)

    def ld(p):
        stem = os.path.splitext(os.path.basename(p))[0]
        x = np.asarray(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))
        m = np.asarray(Image.open(gts[stem]).convert("L").resize((size, size), Image.NEAREST)) > 127
        return x, m

    with ThreadPoolExecutor(8) as ex:
        r = list(ex.map(ld, ims))
    x = np.stack([a for a, _ in r]); m = np.stack([b for _, b in r]).astype(np.uint8)
    np.savez(cache, x=x, m=m)
    return x, m


def synthetic(n, size):
    x = np.random.randint(0, 255, (n, size, size, 3), dtype=np.uint8)
    m = np.zeros((n, size, size), np.uint8)
    for i in range(n):
        a, b = np.random.randint(8, size // 2, 2); m[i, a:a + size // 4, b:b + size // 4] = 1
    return x, m


def get_batch(x, m, idx, dev):
    rgb = torch.from_numpy(x[idx]).to(dev).permute(0, 3, 1, 2).float() / 255
    mk = torch.from_numpy(m[idx]).to(dev).unsqueeze(1).float()
    return rgb, mk


# ------------------------------------------------------------------ train / eval
def loss_fn(logits, mk, gates, mode):
    bce = Fn.binary_cross_entropy_with_logits(logits, mk)
    p = torch.sigmoid(logits)
    iou = 1 - ((p * mk).sum((1, 2, 3)) + 1) / ((p + mk - p * mk).sum((1, 2, 3)) + 1)
    loss = bce + iou.mean()
    if mode == "full" and gates:
        lg = sum(-0.5 * ((ar - 0.5) ** 2 + (ai - 0.5) ** 2).mean() for ar, ai in gates) / len(gates)
        loss = loss + 0.05 * lg
    return loss


@torch.no_grad()
def evaluate(model, x, m, dev, bs=32):
    model.eval(); res = {}
    for s in (0, 1):
        for d in DELTAS:
            gen = torch.Generator(device=dev).manual_seed(1234)
            iou = f1 = mae = n = 0.0; ar = [0.0] * 3; ai = [0.0] * 3
            for i in range(0, len(x), bs):
                idx = np.arange(i, min(i + bs, len(x)))
                rgb, mk = get_batch(x, m, idx, dev)
                ir = simulate_lwir(rgb, mk, torch.full((len(idx),), float(d)), gen)
                if s:
                    rgb = shadow_rgb(rgb, gen)
                logits, gates = model(rgb, ir)
                p = torch.sigmoid(logits); pb = (p > 0.5).float()
                inter = (pb * mk).sum((1, 2, 3)); uni = (pb + mk - pb * mk).sum((1, 2, 3))
                iou += (inter / (uni + 1e-6)).sum().item()
                f1 += (2 * inter / (pb.sum((1, 2, 3)) + mk.sum((1, 2, 3)) + 1e-6)).sum().item()
                mae += (p - mk).abs().mean((1, 2, 3)).sum().item(); n += len(idx)
                for j, (a_r, a_i) in enumerate(gates):
                    ar[j] += a_r.sum().item(); ai[j] += a_i.sum().item()
            res[f"d{d}_s{s}"] = dict(iou=iou / n, f1=f1 / n, mae=mae / n,
                                     alpha_rgb=[v / n for v in ar], alpha_ir=[v / n for v in ai])
            print(f"  delta={d} shadow={s}  IoU={iou/n:.4f}  F1={f1/n:.4f}  MAE={mae/n:.4f}  alpha_ir={[round(v/n,3) for v in ai]}", flush=True)
    return res


@torch.no_grad()
def latency(model, size, dev):
    model.eval(); x = torch.rand(1, 3, size, size, device=dev); y = torch.rand(1, 1, size, size, device=dev)
    for _ in range(10): model(x, y)
    if dev.type == "cuda": torch.cuda.synchronize()
    t = time.time()
    for _ in range(50): model(x, y)
    if dev.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / 50 * 1000


def train(a):
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if a.smoke:
        a.size, a.epochs, a.bs = 96, 1, 8
        xtr, mtr = synthetic(32, a.size); xte, mte = synthetic(16, a.size)
    else:
        xtr, mtr = load_split(a.root, "Train", a.size); xte, mte = load_split(a.root, "Test", a.size)
    print(f"train {len(xtr)}  test {len(xte)}  device {dev}", flush=True)
    model = Net(a.mode, pretrained=not a.smoke, extra=a.raw_desc).to(dev)
    nparam = sum(p.numel() for p in model.parameters()) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    steps = a.epochs * (len(xtr) // a.bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=steps, pct_start=0.1)
    use_amp = dev.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    t0 = time.time(); it = 0
    for ep in range(a.epochs):
        model.train(); perm = np.random.permutation(len(xtr))
        for i in range(0, len(xtr) - a.bs + 1, a.bs):
            idx = np.sort(perm[i:i + a.bs])
            rgb, mk = get_batch(xtr, mtr, idx, dev)
            if random.random() < 0.5:
                rgb, mk = rgb.flip(-1), mk.flip(-1)
            delta = torch.tensor(np.random.choice(TRAIN_D, a.bs), dtype=torch.float32)
            ir = simulate_lwir(rgb, mk, delta)
            sh = (torch.rand(a.bs, device=dev) < 0.3).view(-1, 1, 1, 1)
            rgb = torch.where(sh, shadow_rgb(rgb), rgb)
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                logits, gates = model(rgb, ir)
            loss = loss_fn(logits.float(), mk, gates, a.mode)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); it += 1
            if it % 50 == 0:
                print(f"ep {ep} it {it}/{steps} loss {loss.item():.4f}  {time.time()-t0:.0f}s", flush=True)
    print("evaluating ...", flush=True)
    res = evaluate(model, xte, mte, dev)
    out = dict(mode=a.mode, seed=a.seed, epochs=a.epochs, size=a.size, params_M=nparam,
               latency_ms_bs1=latency(model, a.size, dev), device=str(torch.cuda.get_device_name(0) if use_amp else "cpu"),
               raw_desc=a.raw_desc, res=res)
    name = f"results_{a.mode}{'_raw' if a.raw_desc else ''}_seed{a.seed}.json"
    json.dump(out, open("smoke_" + name if a.smoke else name, "w"), indent=1)
    print("saved", name)


# ------------------------------------------------------------------ report
def report():
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    R = {}
    for f in glob.glob("results_*_seed*.json"):
        d = json.load(open(f)); R.setdefault(d["mode"] + ("_raw" if d.get("raw_desc") else ""), []).append(d)
    names = {"rgb": "RGB only", "concat": "Early concatenation", "fixed": r"Fixed gates ($\alpha=0.5$)",
             "full": "SpectralGate-Net", "full_raw": "SpectralGate-Net + raw descriptor"}
    order = [k for k in names if k in R]

    def st(k, key, fld="iou"):
        v = np.array([r["res"][key][fld] for r in R[k]]) * 100
        return v.mean(), v.std()

    cell = lambda k, key: (f"{st(k,key)[0]:.1f}" if len(R[k]) == 1 else f"{st(k,key)[0]:.1f}$\\pm${st(k,key)[1]:.1f}")
    for s, tag in ((0, "clean"), (1, "shadow")):
        lines = [r"\begin{tabular}{l" + "c" * len(DELTAS) + "}", r"\toprule",
                 "Method & " + " & ".join(rf"$\delta={d}$" for d in DELTAS) + r" \\", r"\midrule"]
        for k in order:
            lines.append(names[k] + " & " + " & ".join(cell(k, f"d{d}_s{s}") for d in DELTAS) + r" \\")
        lines += [r"\bottomrule", r"\end{tabular}"]
        open(f"table_sim_{tag}.tex", "w").write("\n".join(lines))
    eff = [r"\begin{tabular}{lcc}", r"\toprule", r"Method & Params (M) & Latency (ms, bs=1) \\", r"\midrule"]
    for k in order:
        eff.append(f"{names[k]} & {R[k][0]['params_M']:.1f} & {R[k][0]['latency_ms_bs1']:.1f} \\\\")
    eff += [r"\bottomrule", r"\end{tabular}"]
    open("table_eff.tex", "w").write("\n".join(eff))
    print("device:", R[order[0]][0]["device"])

    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    for s in (0, 1):
        for k in order:
            ax[s].plot(DELTAS, [st(k, f"d{d}_s{s}")[0] for d in DELTAS], marker="o", label=k)
        ax[s].set_xlabel(r"thermal contrast $\delta$ (K)"); ax[s].set_ylabel("mIoU (%)")
        ax[s].set_title("clean RGB" if s == 0 else "RGB with shadow"); ax[s].invert_xaxis()
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig("fig_iou_vs_delta.pdf")

    gk = [k for k in ("full", "full_raw") if k in R]
    if gk:
        fig, ax = plt.subplots(figsize=(4, 3))
        for k in gk:
            for s in (0, 1):
                for j, stg in enumerate(("P3", "P4", "P5")):
                    y = [np.mean([r["res"][f"d{d}_s{s}"]["alpha_ir"][j] for r in R[k]]) for d in DELTAS]
                    ax.plot(DELTAS, y, marker="o", ls="-" if s == 0 else "--", label=f"{k} {stg} {'shadow' if s else 'clean'}")
        ax.set_xlabel(r"$\delta$ (K)"); ax.set_ylabel(r"mean $\alpha_{ir}$"); ax.invert_xaxis(); ax.legend(fontsize=5)
        plt.tight_layout(); plt.savefig("fig_alpha_vs_delta.pdf")
    print("wrote table_sim_clean.tex table_sim_shadow.tex table_eff.tex fig_iou_vs_delta.pdf fig_alpha_vs_delta.pdf")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["train", "report"])
    ap.add_argument("--mode", default="full", choices=["rgb", "concat", "fixed", "full"])
    ap.add_argument("--root", default="COD10K-v3")
    ap.add_argument("--size", type=int, default=352)
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--raw_desc", action="store_true", help="add raw-input contrast as 3rd gate feature")
    ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    train(a) if a.cmd == "train" else report()


Writing sgn.py


In [5]:
!python sgn.py train --mode full --smoke

train 32  test 16  device cuda
/kaggle/working/sgn.py:328: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/sgn.py:345: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); it += 1
evaluating ...
  delta=8 shadow=0  IoU=0.0642  F1=0.1207  MAE=0.5482  alpha_ir=[0.5, 0.5, 0.5]
  delta=4 shadow=0  IoU=0.0643  F1=0.1208  MAE=0.5479  alpha_ir=[0.5, 0.5, 0.5]
  delta=2 shadow=0  IoU=0.0642  F1=0.1207  MAE=0.5477  alpha_ir=[0.5, 0.5, 0.

In [6]:
!python sgn.py train --mode full --root /kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3

Train: 3040 camouflaged images
Test: 2026 camouflaged images
train 3040  test 2026  device cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 159MB/s]
/kaggle/working/sgn.py:328: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/sgn.py:345: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.updat

In [7]:
%%bash
R=/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3
cd /kaggle/working

# Launch GPU 0 queue (3 wavelet runs)
nohup sh -c "CUDA_VISIBLE_DEVICES=0 python sgn.py train --mode fixed  --root $R; \
             CUDA_VISIBLE_DEVICES=0 python sgn.py train --mode full  --seed 1 --root $R; \
             CUDA_VISIBLE_DEVICES=0 python sgn.py train --mode fixed --seed 1 --root $R" > gpu0.log 2>&1 &
PID0=$!

# Launch GPU 1 queue (4 baseline runs)
nohup sh -c "CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode rgb    --root $R; \
             CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode concat --root $R; \
             CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode rgb    --seed 1 --root $R; \
             CUDA_VISIBLE_DEVICES=1 python sgn.py train --mode concat --seed 1 --root $R" > gpu1.log 2>&1 &
PID1=$!

echo "Started GPU 0 (PID $PID0) and GPU 1 (PID $PID1). Monitoring every 30s..."

# Print live progress until both GPUs finish
while kill -0 $PID0 2>/dev/null || kill -0 $PID1 2>/dev/null; do
    sleep 30
    echo "=== [$(date +%H:%M:%S)] Status ==="
    echo "[GPU 0]: $(tail -n 1 gpu0.log 2>/dev/null)"
    echo "[GPU 1]: $(tail -n 1 gpu1.log 2>/dev/null)"
done

echo "All 8 runs complete! Generating LaTeX tables and PDF figures..."
python sgn.py report
head -n 50 table_sim_clean.tex table_sim_shadow.tex table_eff.tex

Process is interrupted.


In [8]:
%cd /kaggle/working
!ls -la sgn.py results_*.json
!grep -c lgate sgn.py
!ps aux | grep "sgn.py" | grep -v grep | wc -l
!tail -n 2 gpu0.log gpu1.log

/kaggle/working
-rw-r--r-- 1 root root  2244 Oct 21 09:19 results_concat_seed0.json
-rw-r--r-- 1 root root  2245 Oct 21 09:31 results_concat_seed1.json
-rw-r--r-- 1 root root  2246 Oct 21 09:18 results_fixed_seed0.json
-rw-r--r-- 1 root root  3161 Oct 21 08:31 results_full_seed0.json
-rw-r--r-- 1 root root  3138 Oct 21 09:31 results_full_seed1.json
-rw-r--r-- 1 root root  2238 Oct 21 09:11 results_rgb_seed0.json
-rw-r--r-- 1 root root  2238 Oct 21 09:23 results_rgb_seed1.json
-rw-r--r-- 1 root root 20868 Oct 21 08:12 sgn.py
0
2
==> gpu0.log <==
ep 2 it 500/2280 loss 0.7280  104s
ep 2 it 550/2280 loss 0.9243  114s

==> gpu1.log <==
  delta=0 shadow=1  IoU=0.4145  F1=0.5373  MAE=0.0712  alpha_ir=[0.0, 0.0, 0.0]
saved results_concat_seed1.json


In [9]:
%cd /kaggle/working
s = open('sgn.py').read()
def rep(a, b):
    global s
    assert a in s, "NOT FOUND: " + a
    s = s.replace(a, b, 1)
rep('def loss_fn(logits, mk, gates, mode):', 'def loss_fn(logits, mk, gates, mode, lgate=0.05):')
rep('loss = loss + 0.05 * lg', 'loss = loss + lgate * lg')
rep('loss = loss_fn(logits.float(), mk, gates, a.mode)', 'loss = loss_fn(logits.float(), mk, gates, a.mode, a.lgate)')
rep('raw_desc=a.raw_desc, res=res)', 'raw_desc=a.raw_desc, lgate=a.lgate, res=res)')
rep("""name = f"results_{a.mode}{'_raw' if a.raw_desc else ''}_seed{a.seed}.json\"""",
    """name = f"results_{a.mode}{'_raw' if a.raw_desc else ''}{'_nolg' if a.lgate == 0 else ''}_seed{a.seed}.json\"""")
rep('ap.add_argument("--smoke", action="store_true")',
    'ap.add_argument("--lgate", type=float, default=0.05)\n    ap.add_argument("--smoke", action="store_true")')
rep('R.setdefault(d["mode"] + ("_raw" if d.get("raw_desc") else ""), []).append(d)',
    'R.setdefault(d["mode"] + ("_raw" if d.get("raw_desc") else "") + ("_nolg" if d.get("lgate", 0.05) == 0 else ""), []).append(d)')
rep('"full": "SpectralGate-Net", "full_raw": "SpectralGate-Net + raw descriptor"}',
    '"full": "SpectralGate-Net", "full_nolg": r"SpectralGate-Net, no $L_{gate}$",\n             "full_raw": "SpectralGate-Net + raw descriptor", "full_raw_nolg": r"SpectralGate-Net + raw desc., no $L_{gate}$"}')
rep('gk = [k for k in ("full", "full_raw") if k in R]',
    'gk = [k for k in ("full", "full_nolg", "full_raw", "full_raw_nolg") if k in R]')
open('sgn.py', 'w').write(s)
print("patched")

/kaggle/working
patched


In [10]:
%%bash
cd /kaggle/working
R=/kaggle/input/datasets/ivanomelchenkoim11/cod10k-dataset/COD10K-v3
# GPU 1 is free now: no polarisation loss
nohup sh -c "CUDA_VISIBLE_DEVICES=1 python -u sgn.py train --mode full --lgate 0 --root $R" > gpu1b.log 2>&1 &
# GPU 0: wait for the fixed-seed1 job to end, then run no-L_gate + raw descriptor
nohup sh -c "while pgrep -f '[s]gn.py train --mode fixed --seed 1' >/dev/null; do sleep 10; done; CUDA_VISIBLE_DEVICES=0 python -u sgn.py train --mode full --lgate 0 --raw_desc --root $R" > gpu0b.log 2>&1 &
sleep 2; echo launched

launched


In [11]:
!python sgn.py report
!cat table_sim_clean.tex table_sim_shadow.tex

device: Tesla T4
wrote table_sim_clean.tex table_sim_shadow.tex table_eff.tex fig_iou_vs_delta.pdf fig_alpha_vs_delta.pdf
\begin{tabular}{lccccc}
\toprule
Method & $\delta=8$ & $\delta=4$ & $\delta=2$ & $\delta=1$ & $\delta=0$ \\
\midrule
RGB only & 47.3$\pm$0.0 & 47.3$\pm$0.0 & 47.3$\pm$0.0 & 47.3$\pm$0.0 & 47.3$\pm$0.0 \\
Early concatenation & 88.1$\pm$0.0 & 87.9$\pm$0.1 & 87.0$\pm$0.1 & 81.1$\pm$0.0 & 42.5$\pm$0.6 \\
Fixed gates ($\alpha=0.5$) & 86.1 & 85.9 & 84.7 & 77.5 & 41.9 \\
SpectralGate-Net & 85.2$\pm$1.3 & 85.0$\pm$1.2 & 83.9$\pm$1.3 & 77.2$\pm$1.4 & 42.5$\pm$0.5 \\
\bottomrule
\end{tabular}\begin{tabular}{lccccc}
\toprule
Method & $\delta=8$ & $\delta=4$ & $\delta=2$ & $\delta=1$ & $\delta=0$ \\
\midrule
RGB only & 45.6$\pm$0.0 & 45.6$\pm$0.0 & 45.6$\pm$0.0 & 45.6$\pm$0.0 & 45.6$\pm$0.0 \\
Early concatenation & 88.1$\pm$0.1 & 87.9$\pm$0.1 & 86.9$\pm$0.1 & 80.7$\pm$0.0 & 40.7$\pm$0.7 \\
Fixed gates ($\alpha=0.5$) & 86.1 & 85.9 & 84.7 & 77.4 & 40.0 \\
SpectralGate-Net & 85.2$

In [12]:
!tail -n 1 gpu0.log gpu0b.log gpu1b.log
!ls results_*.json

==> gpu0.log <==
  delta=8 shadow=1  IoU=0.8646  F1=0.9213  MAE=0.0135  alpha_ir=[0.5, 0.5, 0.5]

==> gpu0b.log <==

==> gpu1b.log <==
ep 5 it 1050/2280 loss 0.6225  260s
results_concat_seed0.json  results_full_seed0.json  results_rgb_seed1.json
results_concat_seed1.json  results_full_seed1.json
results_fixed_seed0.json   results_rgb_seed0.json


In [13]:
import json, glob
D = [8, 4, 2, 1, 0]
for f in sorted(glob.glob('results_*.json')):
    r = json.load(open(f))['res']
    print(f.replace('results_', '').replace('.json', '').ljust(22),
          'IoU%', [round(r[f'd{d}_s0']['iou'] * 100, 1) for d in D],
          '| alpha_ir d8:', [round(a, 2) for a in r['d8_s0']['alpha_ir']],
          'd0:', [round(a, 2) for a in r['d0_s0']['alpha_ir']])

concat_seed0           IoU% [88.2, 87.9, 87.1, 81.1, 41.9] | alpha_ir d8: [0.0, 0.0, 0.0] d0: [0.0, 0.0, 0.0]
concat_seed1           IoU% [88.1, 87.8, 86.9, 81.0, 43.1] | alpha_ir d8: [0.0, 0.0, 0.0] d0: [0.0, 0.0, 0.0]
fixed_seed0            IoU% [86.1, 85.9, 84.7, 77.5, 41.9] | alpha_ir d8: [0.5, 0.5, 0.5] d0: [0.5, 0.5, 0.5]
full_seed0             IoU% [83.9, 83.8, 82.6, 75.9, 42.0] | alpha_ir d8: [0.98, 0.96, 0.61] d0: [0.97, 0.96, 0.6]
full_seed1             IoU% [86.5, 86.3, 85.2, 78.6, 43.0] | alpha_ir d8: [0.99, 0.97, 0.62] d0: [0.98, 0.98, 0.6]
rgb_seed0              IoU% [47.3, 47.3, 47.3, 47.3, 47.3] | alpha_ir d8: [0.0, 0.0, 0.0] d0: [0.0, 0.0, 0.0]
rgb_seed1              IoU% [47.2, 47.2, 47.2, 47.2, 47.2] | alpha_ir d8: [0.0, 0.0, 0.0] d0: [0.0, 0.0, 0.0]


In [14]:
!ls results_*.json
!echo "---- gpu1b.log (no L_gate) ----"; tail -n 12 gpu1b.log
!echo "---- gpu0b.log (no L_gate + raw desc) ----"; tail -n 12 gpu0b.log
!ps aux | grep "sgn.py" | grep -v grep | wc -l

results_concat_seed0.json     results_full_raw_nolg_seed0.json
results_concat_seed1.json     results_full_seed0.json
results_fixed_seed0.json      results_full_seed1.json
results_fixed_seed1.json      results_rgb_seed0.json
results_full_nolg_seed0.json  results_rgb_seed1.json
---- gpu1b.log (no L_gate) ----
evaluating ...
  delta=8 shadow=0  IoU=0.8609  F1=0.9192  MAE=0.0148  alpha_ir=[0.639, 0.692, 0.575]
  delta=4 shadow=0  IoU=0.8587  F1=0.9176  MAE=0.0151  alpha_ir=[0.64, 0.694, 0.575]
  delta=2 shadow=0  IoU=0.8463  F1=0.9091  MAE=0.0165  alpha_ir=[0.641, 0.696, 0.576]
  delta=1 shadow=0  IoU=0.7741  F1=0.8554  MAE=0.0246  alpha_ir=[0.646, 0.698, 0.575]
  delta=0 shadow=0  IoU=0.4178  F1=0.5390  MAE=0.0713  alpha_ir=[0.652, 0.688, 0.569]
  delta=8 shadow=1  IoU=0.8601  F1=0.9183  MAE=0.0149  alpha_ir=[0.639, 0.693, 0.575]
  delta=4 shadow=1  IoU=0.8578  F1=0.9167  MAE=0.0152  alpha_ir=[0.639, 0.695, 0.575]
  delta=2 shadow=1  IoU=0.8449  F1=0.9078  MAE=0.0168  alpha_ir=[0.641, 0.6

In [15]:
!python sgn.py report
import json, glob
D = [8, 4, 2, 1, 0]
for f in sorted(glob.glob('results_*.json')):
    r = json.load(open(f))['res']
    print(f.replace('results_','').replace('.json','').ljust(24),
          'IoU%', [round(r[f'd{d}_s0']['iou']*100,1) for d in D],
          '| alpha_ir d8:', [round(a,2) for a in r['d8_s0']['alpha_ir']],
          'd0:', [round(a,2) for a in r['d0_s0']['alpha_ir']])

device: Tesla T4
wrote table_sim_clean.tex table_sim_shadow.tex table_eff.tex fig_iou_vs_delta.pdf fig_alpha_vs_delta.pdf
concat_seed0             IoU% [88.2, 87.9, 87.1, 81.1, 41.9] | alpha_ir d8: [0.0, 0.0, 0.0] d0: [0.0, 0.0, 0.0]
concat_seed1             IoU% [88.1, 87.8, 86.9, 81.0, 43.1] | alpha_ir d8: [0.0, 0.0, 0.0] d0: [0.0, 0.0, 0.0]
fixed_seed0              IoU% [86.1, 85.9, 84.7, 77.5, 41.9] | alpha_ir d8: [0.5, 0.5, 0.5] d0: [0.5, 0.5, 0.5]
fixed_seed1              IoU% [86.6, 86.3, 85.2, 78.7, 43.2] | alpha_ir d8: [0.5, 0.5, 0.5] d0: [0.5, 0.5, 0.5]
full_nolg_seed0          IoU% [86.1, 85.9, 84.6, 77.4, 41.8] | alpha_ir d8: [0.64, 0.69, 0.58] d0: [0.65, 0.69, 0.57]
full_raw_nolg_seed0      IoU% [86.9, 86.7, 85.6, 78.8, 41.8] | alpha_ir d8: [0.83, 0.62, 0.6] d0: [0.77, 0.63, 0.59]
full_seed0               IoU% [83.9, 83.8, 82.6, 75.9, 42.0] | alpha_ir d8: [0.98, 0.96, 0.61] d0: [0.97, 0.96, 0.6]
full_seed1               IoU% [86.5, 86.3, 85.2, 78.6, 43.0] | alpha_ir d8: [0

In [17]:
import glob, zipfile
with zipfile.ZipFile('paper_outputs.zip', 'w') as z:
    for p in glob.glob('*.json') + glob.glob('*.tex') + glob.glob('*.pdf') + glob.glob('*.log') + ['sgn.py']:
        if 'smoke' not in p:
            z.write(p)

In [18]:
import json, matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
D = [8, 4, 2, 1, 0]
runs = {r'with $L_{gate}$': 'results_full_seed0.json', r'no $L_{gate}$': 'results_full_nolg_seed0.json',
        r'no $L_{gate}$ + raw desc.': 'results_full_raw_nolg_seed0.json'}
fig, ax = plt.subplots(1, 3, figsize=(9, 2.6), sharey=True)
for j, st in enumerate(['P3', 'P4', 'P5']):
    for n, f in runs.items():
        r = json.load(open(f))['res']
        ax[j].plot(D, [r[f'd{d}_s0']['alpha_ir'][j] for d in D], marker='o', label=n)
    ax[j].set_title(st); ax[j].set_xlabel(r'$\delta$ (K)'); ax[j].invert_xaxis()
ax[0].set_ylabel(r'mean $\alpha_{ir}$'); ax[0].legend(fontsize=6); plt.tight_layout(); plt.savefig('fig_alpha_clean.pdf')

In [19]:
import torch
from sgn import dwt, idwt
x = torch.randn(2, 8, 16, 16)
print((idwt(*dwt(x)) - x).abs().max().item())
print(abs(sum((b**2).sum() for b in dwt(x)).item() - (x**2).sum().item()))

2.384185791015625e-07
0.0


In [20]:
import json
for f in ['results_full_seed0.json', 'results_full_seed1.json']:
    print(f, [round(a, 2) for a in json.load(open(f))['res']['d8_s0']['alpha_rgb']])

results_full_seed0.json [0.04, 0.58, 0.48]
results_full_seed1.json [0.52, 0.79, 0.48]


In [21]:
!pip list 2>/dev/null | grep -i -E "^(torch|torchvision|numpy|pillow|matplotlib) "

matplotlib                            3.10.0
numpy                                 2.1.3
pillow                                12.3.0
torch                                 2.11.0+cu128
torchvision                           0.26.0+cu128
